In [1]:
!pip install xgboost transformers torch torchvision scikit-learn pillow


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\sulta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import json
import math
from pathlib import Path

import numpy as np
from PIL import Image

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb


In [3]:
# Paths (relative to notebooks/)
ROOT = Path('..')
SYN_IMAGES = ROOT / 'Synthetic_dataset' / 'images'
SYN_LABELS = ROOT / 'Synthetic_dataset' / 'labels'
TEST_IMAGES = ROOT / 'Testing' / 'images'
TEST_INPUT_JSON = ROOT / 'Testing' / 'Input_jsons'
TEST_GT = ROOT / 'Testing' / 'Error_bar_groundTruth'
PRED_OUT = ROOT / 'Testing' / 'Predicted_error_bar'
PRED_OUT.mkdir(parents=True, exist_ok=True)

MODEL_TYPE = 'clip'  # 'clip' or 'dinov2'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Device:', DEVICE)


Device: cpu


In [4]:
def load_clip():
    from transformers import CLIPProcessor, CLIPModel
    model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    model.to(DEVICE).eval()
    return model, processor


def load_dinov2():
    # Requires internet the first time to download weights
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
    model.to(DEVICE).eval()
    return model


In [5]:
def build_image_embedder(model_type):
    if model_type == 'clip':
        model, processor = load_clip()

        def embed(img):
            inputs = processor(images=img, return_tensors='pt')
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                try:
                    feats = model.get_image_features(**inputs)
                except Exception:
                    out = model(**inputs)
                    if hasattr(out, 'image_embeds') and out.image_embeds is not None:
                        feats = out.image_embeds
                    elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
                        feats = out.pooler_output
                    elif hasattr(out, 'last_hidden_state'):
                        feats = out.last_hidden_state[:, 0]
                    else:
                        feats = out[0]
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
            if hasattr(feats, 'pooler_output') and feats.pooler_output is not None:
                feats = feats.pooler_output
            if hasattr(feats, 'last_hidden_state'):
                feats = feats.last_hidden_state[:, 0]
            if not torch.is_tensor(feats):
                feats = torch.tensor(feats)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            return feats.squeeze(0).cpu().numpy()

        return embed

    if model_type == 'dinov2':
        model = load_dinov2()
        from torchvision import transforms
        transform = transforms.Compose([
            transforms.Resize(224),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        def embed(img):
            x = transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                feats = model(x)
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
            if hasattr(feats, 'pooler_output') and feats.pooler_output is not None:
                feats = feats.pooler_output
            if hasattr(feats, 'last_hidden_state'):
                feats = feats.last_hidden_state[:, 0]
            if not torch.is_tensor(feats):
                feats = torch.tensor(feats)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            return feats.squeeze(0).cpu().numpy()

        return embed

    raise ValueError('MODEL_TYPE must be clip or dinov2')


embed_image = build_image_embedder(MODEL_TYPE)


C:\Users\sulta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 299.31it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the m

In [6]:
def load_json(path):
    return json.loads(Path(path).read_text())


def get_axis_points(points):
    xmin = next((p for p in points if p.get('label') == 'xmin'), None)
    xmax = next((p for p in points if p.get('label') == 'xmax'), None)
    ymin = next((p for p in points if p.get('label') == 'ymin'), None)
    ymax = next((p for p in points if p.get('label') == 'ymax'), None)
    return xmin, xmax, ymin, ymax


def axis_from_points(points):
    xs = [p['x'] for p in points]
    ys = [p['y'] for p in points]
    return min(xs), max(xs), min(ys), max(ys)


def point_features_from_axis(pt, xmin, xmax, ymin, ymax, img_w, img_h, series_idx, point_idx):
    if xmin is not None and xmax is not None and xmax != xmin:
        x_norm = (pt['x'] - xmin) / (xmax - xmin)
    else:
        x_norm = pt['x'] / img_w
    if ymin is not None and ymax is not None and ymin != ymax:
        y_norm = (pt['y'] - ymax) / (ymin - ymax)
    else:
        y_norm = pt['y'] / img_h
    return np.array([x_norm, y_norm, series_idx, point_idx], dtype=np.float32)


def point_features(pt, xmin, xmax, ymin, ymax, img_w, img_h, series_idx, point_idx):
    # xmin/xmax/ymin/ymax are label points (dicts)
    if xmin and xmax and xmax['x'] != xmin['x']:
        x_norm = (pt['x'] - xmin['x']) / (xmax['x'] - xmin['x'])
    else:
        x_norm = pt['x'] / img_w
    if ymin and ymax and ymin['y'] != ymax['y']:
        y_norm = (pt['y'] - ymax['y']) / (ymin['y'] - ymax['y'])
    else:
        y_norm = pt['y'] / img_h
    return np.array([x_norm, y_norm, series_idx, point_idx], dtype=np.float32)


def build_training_data(images_dir, labels_dir):
    X = []
    y = []
    for label_path in labels_dir.glob('*.json'):
        image_path = images_dir / f'{label_path.stem}.png'
        if not image_path.exists():
            continue
        img = Image.open(image_path).convert('RGB')
        img_w, img_h = img.size
        img_feat = embed_image(img)
        data = load_json(label_path)
        for s_idx, series in enumerate(data):
            xmin, xmax, ymin, ymax = get_axis_points(series['points'])
            point_idx = 0
            for pt in series['points']:
                if pt['label'] != '':
                    continue
                pf = point_features(pt, xmin, xmax, ymin, ymax, img_w, img_h, s_idx, point_idx)
                point_idx += 1
                feats = np.concatenate([img_feat, pf])
                X.append(feats)
                y.append([pt['topBarPixelDistance'], pt['bottomBarPixelDistance'], pt['deviationPixelDistance']])
    return np.array(X), np.array(y)


In [7]:
# Build training data from Synthetic_dataset
X, y = build_training_data(SYN_IMAGES, SYN_LABELS)
print('Train samples:', X.shape, y.shape)


Train samples: (152004, 516) (152004, 3)


In [8]:
# Train XGBoost (multi-output)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
base = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
)
model = MultiOutputRegressor(base)
model.fit(X_train, y_train)

val_pred = model.predict(X_val)
val_mae = np.mean(np.abs(val_pred - y_val), axis=0)
print('Val MAE [top, bottom, deviation]:', val_mae)


Val MAE [top, bottom, deviation]: [11.31613118 11.70720666 10.89677679]


In [9]:
def predict_for_testing():
    for input_path in TEST_INPUT_JSON.glob('*.json'):
        image_path = TEST_IMAGES / f'{input_path.stem}.png'
        if not image_path.exists():
            continue
        img = Image.open(image_path).convert('RGB')
        img_w, img_h = img.size
        img_feat = embed_image(img)
        data = load_json(input_path)

        if isinstance(data, dict) and 'data_points' in data:
            series_list = data['data_points']
        else:
            series_list = data

        out = []
        for s_idx, series in enumerate(series_list):
            points = series.get('points', [])
            if not points:
                continue
            xmin_v, xmax_v, ymin_v, ymax_v = axis_from_points(points)
            point_idx = 0
            points_out = []
            for pt in points:
                pf = point_features_from_axis(pt, xmin_v, xmax_v, ymin_v, ymax_v, img_w, img_h, s_idx, point_idx)
                point_idx += 1
                feats = np.concatenate([img_feat, pf])[None, :]
                pred = model.predict(feats)[0]
                top_p, bottom_p, _dev_p = [float(max(0.0, v)) for v in pred]
                points_out.append({
                    'data_point': {'x': pt['x'], 'y': pt['y']},
                    'upper_error_bar': {'x': pt['x'], 'y': pt['y'] - top_p},
                    'lower_error_bar': {'x': pt['x'], 'y': pt['y'] + bottom_p},
                })
            out.append({'lineName': series.get('lineName', f'Series_{s_idx+1}'), 'points': points_out})

        pred_obj = {'image_file': f'{input_path.stem}.png', 'error_bars': out}
        (PRED_OUT / f'{input_path.stem}.json').write_text(json.dumps(pred_obj, indent=2))

predict_for_testing()
print('Predictions saved to:', PRED_OUT)


Predictions saved to: ..\Testing\Predicted_error_bar


In [10]:
# Evaluate: compare ground-truth vs predicted
TOL = 5.0  # pixel tolerance
y_true = []
y_pred = []

def extract_top_bottom_from_errorbars(series):
    vals = []
    for p in series.get('points', []):
        dp = p['data_point']
        up = p['upper_error_bar']
        low = p['lower_error_bar']
        top = max(0.0, dp['y'] - up['y'])
        bottom = max(0.0, low['y'] - dp['y'])
        vals.append([top, bottom])
    return vals

for gt_path in TEST_GT.glob('*.json'):
    pred_path = PRED_OUT / f'{gt_path.stem}.json'
    if not pred_path.exists():
        continue
    gt_obj = load_json(gt_path)
    pr_obj = load_json(pred_path)
    gt_series = gt_obj.get('error_bars', [])
    pr_series = pr_obj.get('error_bars', [])
    for s_gt, s_pr in zip(gt_series, pr_series):
        gt_vals = extract_top_bottom_from_errorbars(s_gt)
        pr_vals = extract_top_bottom_from_errorbars(s_pr)
        for g, p in zip(gt_vals, pr_vals):
            y_true.append(g)
            y_pred.append(p)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

diff = np.abs(y_true - y_pred)
correct = (diff[:, 0] <= TOL) & (diff[:, 1] <= TOL)
pred_labels = correct.astype(int)
true_labels = np.ones_like(pred_labels)

acc = accuracy_score(true_labels, pred_labels)
cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1])
print('Accuracy (within tolerance):', acc)
print('Confusion matrix [0=incorrect,1=correct]:')
print(cm)

mae = np.mean(np.abs(y_true - y_pred), axis=0)
print('MAE [top, bottom]:', mae)


Accuracy (within tolerance): 0.022938623682579044
Confusion matrix [0=incorrect,1=correct]:
[[   0    0]
 [4728  111]]
MAE [top, bottom]: [31.10675278 37.46532478]
